# Group B2: Lab Test – Preprocessing, Embedding MLP, and Model Evaluation

### Task Objectives:
1. **Preprocessing**: Implement a vocabulary builder and sentence padding function.
2. **Model Architecture**: Construct an `EmbeddingMLP` using `nn.Embedding` + Mean Pooling + Linear Layer.
3. **Evaluation**: Write the evaluation function to compute classification accuracy on a validation dataset.

In [ ]:
import torchttorchtorchtorchorchtorch
torchtorchtorch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

# Raw Dataset
raw_train_data = [
    ("win free cash prize now", 1),
    ("claim your free gift today", 1),
    ("urgent offer free money inside", 1),
    ("project status report attached", 0),
    ("meeting rescheduled to tomorrow morning", 0),
    ("please review the attached document", 0)
]

raw_test_data = [
    ("free cash offer inside", 1),
    ("please review the project report", 0)
]

# =============================================================================
# PART 1: PREPROCESSING (Student TODO)
# =============================================================================
def build_vocab(corpus):
    """
    TODO: Build a dictionary mapping unique words to integer IDs.
    Must reserve: '<PAD>': 0, '<UNK>': 1
    """
    vocab = {"<PAD>": 0, "<UNK>": 1}
    idx = 2
    for sentence in corpus:
        for word in sentence.lower().split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

def encode_and_pad(sentence, vocab, max_len=6):
    """
    TODO: Convert sentence string to list of token IDs padded/truncated to max_len.
    """
    words = sentence.lower().split()
    encoded = [vocab.get(word, vocab["<UNK>"]) for word in words]
    if len(encoded) < max_len:
        encoded = encoded + [vocab["<PAD>"]] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return encoded

# =============================================================================
# PART 2: MODEL ARCHITECTURE (Student TODO)
# =============================================================================
class EmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, num_classes=1):
        super().__init__()
        # =========================================================================
        # TODO: Define layers:
        # 1. Embedding layer (padding_idx=0)
        # 2. Linear layer (embed_dim -> hidden_dim) + ReLU
        # 3. Linear layer (hidden_dim -> num_classes)
        # =========================================================================
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # =========================================================================
        # TODO: Implement forward pass with Mean Pooling across sequence dimension
        # =========================================================================
        embedded = self.embedding(x)  # Shape: [batch_size, seq_len, embed_dim]
        pooled = torch.mean(embedded, dim=1)  # Shape: [batch_size, embed_dim]
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.fc2(out)
        return out.squeeze(-1)  # Shape: [batch_size]

# =============================================================================
# PART 3: EVALUATION METRIC (Student TODO)
# =============================================================================
def evaluate_accuracy(model, data_loader):
    """
    TODO: Evaluate the model on data_loader and return accuracy percentage (0.0 to 100.0).
    Convert logits using sigmoid threshold of 0.5.
    """
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for bx, by in data_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == by).sum().item()
            total += by.size(0)
    return (correct / total) * 100 if total > 0 else 0.0

# =============================================================================
# PART 4: AUTOMATED VERIFICATION SUITE (Do Not Modify)
# =============================================================================
def run_test():
    vocab = build_vocab([text for text, _ in raw_train_data])
    assert vocab is not None and len(vocab) > 2, "Preprocessing Failed: Vocabulary not built correctly."

    X_train = torch.tensor([encode_and_pad(t, vocab) for t, _ in raw_train_data], dtype=torch.long)
    y_train = torch.tensor([l for _, l in raw_train_data], dtype=torch.float32)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=2, shuffle=True)

    X_test = torch.tensor([encode_and_pad(t, vocab) for t, _ in raw_test_data], dtype=torch.long)
    y_test = torch.tensor([l for _, l in raw_test_data], dtype=torch.float32)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=2)

    model = EmbeddingMLP(vocab_size=len(vocab)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCEWithLogitsLoss()

    # Train Loop
    model.train()
    for _ in range(20):
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    test_acc = evaluate_accuracy(model, test_loader)
    print(f"Test Set Accuracy: {test_acc:.2f}%")
    assert test_acc == 100.0, "Evaluation Failed: Model did not achieve target accuracy on test set."
    print("Lab task passed!")

run_test()

Test Set Accuracy: 100.00%
Lab task passed!
